# Notebook 02: Clinical Data Preprocessing & Leakage Prevention
This notebook walks through cleaning, temporal feature engineering, longitudinal imputation, categorical encoding, and leak-free train/test scaling.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import config
from preprocessing.load_data import load_raw_data
from preprocessing.cleaning import clean_records
from preprocessing.feature_engineering import engineer_features
from preprocessing.missing_values import LongitudinalImputer
from preprocessing.encoding import CategoricalEncoder
from preprocessing.normalization import ClinicalScaler
from preprocessing.pipeline import PreprocessingPipeline, split_by_patient

## 1. Load Raw Healthcare Records

In [ ]:
raw_df = load_raw_data(source="auto", num_patients=200)
print(f"Raw shape: {raw_df.shape}")
print(f"Missing values before preprocessing:
{raw_df.isna().sum()[raw_df.isna().sum() > 0]}")

## 2. Data Cleaning & Physiological Bounds

In [ ]:
cleaned_df = clean_records(raw_df, min_visits=2)
print(f"Cleaned shape: {cleaned_df.shape}")
cleaned_df.head(5)

## 3. Temporal Feature Engineering (Visit Index, dt)

In [ ]:
engineered_df = engineer_features(cleaned_df)
engineered_df[["subject_id", "chartdate", "visit_number", "days_since_last_visit", "cumulative_days"]].head(8)

## 4. Patient-level Train / Test Split (Zero Cross-Patient Leakage)

In [ ]:
train_df, test_df = split_by_patient(engineered_df, test_size=0.2, seed=42)
print(f"Train patients: {train_df['subject_id'].nunique()}, Records: {len(train_df)}")
print(f"Test patients:  {test_df['subject_id'].nunique()}, Records: {len(test_df)}")

# Assert zero patient intersection
assert len(set(train_df['subject_id']).intersection(set(test_df['subject_id']))) == 0
print("OK Verified zero patient overlap between train and test splits!")

## 5. Fit & Save Imputer, Encoder, and Normalization Scaler

In [ ]:
imputer = LongitudinalImputer().fit(train_df)
train_imputed = imputer.transform(train_df)
test_imputed = imputer.transform(test_df)

encoder = CategoricalEncoder().fit(train_imputed)
train_encoded = encoder.transform(train_imputed)
test_encoded = encoder.transform(test_imputed)

print("Imputed and encoded sample:")
train_encoded[["subject_id", "visit_number", "gender_encoded", "smoking_status_encoded", "hba1c"]].head()